## CLEAN LOADING & ADDING FEATURE ENGINEERING

In [ ]:
import pandas as pd
import pyarrow.parquet as pq

file_path = '/kaggle/input/datasets/sciencekonstant/completeindia/finalwithscore.parquet'

# Get all column names from the file metadata without loading the data
file_meta = pq.read_metadata(file_path)
all_columns = file_meta.schema.names

# Define all the columns you want to exclude
columns_to_drop = {'geometry', '__index_level_0__', 'lat', 'long','nearest_road_surface'}

# Keep everything except the ones in columns_to_drop
columns_to_keep = [col for col in all_columns if col not in columns_to_drop]

# Load only the columns you need
df = pd.read_parquet(file_path, columns=columns_to_keep)

In [ ]:
# Feature 1: The Commercial Hub Index
df['commercial_hub_index'] = df['junction_density_300m'] * df['synergy_services_and_business_1000m']

# Feature 2: The Isolation Penalty
# (Competitor distance * lack of junctions)
df['isolation_penalty'] = df['nearest_comp_dist'] / (df['junction_density_300m'] + 1)

## AUTOMATED FEATURE ENGINEERING , ALREADY KNOWN SO DIRECTLY USED THE FUNCTION

In [ ]:
# !pip install openfe

In [ ]:
# import numpy as np
# import sklearn.metrics
# import pandas as pd

# print("1. Patching Scikit-Learn in memory to protect OpenFE...")

# # -- THE MONKEY PATCH --
# # Save the original Mean Squared Error function
# original_mse = sklearn.metrics.mean_squared_error

# # Create a custom wrapper that catches the broken 'squared=False' argument
# def patched_mse(y_true, y_pred, *args, **kwargs):
#     squared = kwargs.pop('squared', True) # Intercept and remove the bad argument
#     mse = original_mse(y_true, y_pred, *args, **kwargs)
#     return np.sqrt(mse) if not squared else mse

# # Overwrite Scikit-Learn's function in active memory!
# sklearn.metrics.mean_squared_error = patched_mse
# # ----------------------

# # NOW it is safe to import OpenFE
# from openfe import OpenFE, transform

# print("2. Sampling data for robust AutoFE discovery...")
# # Grab 100,000 rows to let the AI experiment quickly
# X_sample = X_train.sample(100000, random_state=42)
# y_sample = y_train.loc[X_sample.index]

# print("3. Launching OpenFE (Testing +, -, *, /, and GroupBys)...")
# ofe = OpenFE()

# # The engine will use tree-based boosting to prove which features actually lower the RMSE
# features = ofe.fit(
#     data=X_sample, 
#     label=y_sample, 
#     task='regression',
#     n_jobs=2,               
#     feature_boosting=True   
# )

# print("\n🏆 Top 15 Advanced Features Discovered:")
# for feat in features[:15]:
#     print(f" -> {feat.name}")

# print("\n4. Applying the Top 15 robust features to the full datasets...")
# # transform() mathematically generates the new columns across your 2.5M rows
# X_train, X_val = transform(X_train, X_val, features[:15], n_jobs=2)
# _, X_test = transform(X_train, X_test, features[:15], n_jobs=2)

# print("✅ Robust Automated Feature Engineering Complete!")

In [ ]:
# def decode_formula(node):
#     # Base case: if it has no children, it's one of your original columns
#     if not getattr(node, 'children', None):
#         return getattr(node, 'name', str(node))
    
#     # Recursive case: it's an operator, so decode its children
#     child_formulas = [decode_formula(c) for c in node.children]
#     op = node.name
    
#     # Format the output beautifully based on how many children the operator has
#     if len(child_formulas) == 2:
#         if op in ['+', '-', '*', '/']:
#             return f"({child_formulas[0]} {op} {child_formulas[1]})"
#         else:
#             return f"{op} of ({child_formulas[0]} grouped by {child_formulas[1]})"
#     elif len(child_formulas) == 1:
#         return f"{op}({child_formulas[0]})"
#     else:
#         return f"{op}({', '.join(child_formulas)})"

# print("🔍 Hacking the AI's Secret Formulas:")
# for i, feat in enumerate(features[:15]):
#     try:
#         print(f"autoFE_f_{i}  -->  {decode_formula(feat)}")
#     except Exception as e:
#         print(f"autoFE_f_{i}  -->  [Complex GroupBy/Target Encoding]")

## FEATURE ENGINEERING FUNCTION

In [ ]:
import pandas as pd
import numpy as np

def generate_top_15_features(df):
    """
    Manually recreates the top 15 features discovered by OpenFE.
    Note: Adds a tiny epsilon (1e-5) to denominators to prevent division-by-zero (inf) errors.
    """
    df_out = df.copy()
    eps = 1e-5 

    # # 1. GroupBy Min Features
    # # Uses .transform('min') to calculate the min and map it back to every row of the same group
    # df_out['autoFE_f_0'] = df_out.groupby('dist_nearest_park')['target_category'].transform('min')
    # df_out['autoFE_f_1'] = df_out.groupby('comp_count_1000m')['target_category'].transform('min')

    # 2. Multipliers & Subtractions
    df_out['autoFE_f_3'] = df_out['synergy_services_and_business_300m'] * df_out['synergy_lodging_1000m']
    df_out['autoFE_f_4'] = df_out['isolation_penalty'] - df_out['synergy_cultural_and_historic_300m']
    df_out['autoFE_f_8'] = df_out['synergy_services_and_business_500m'] * df_out['synergy_geographic_entities_1000m']
    df_out['autoFE_f_9'] = df_out['synergy_lifestyle_services_1000m'] * df_out['junction_density_300m']

    # 3. Ratios (Division)
    df_out['autoFE_f_2'] = df_out['synergy_services_and_business_500m'] / (df_out['synergy_lodging_300m'] + eps)
    df_out['autoFE_f_6'] = df_out['synergy_lifestyle_services_1000m'] / (df_out['synergy_community_and_government_300m'] + eps)
    df_out['autoFE_f_7'] = df_out['synergy_food_and_drink_500m'] / (df_out['synergy_travel_and_transportation_500m'] + eps)
    df_out['autoFE_f_10'] = df_out['synergy_services_and_business_1000m'] / (df_out['synergy_sports_and_recreation_500m'] + eps)
    df_out['autoFE_f_12'] = df_out['synergy_geographic_entities_1000m'] / (df_out['synergy_sports_and_recreation_1000m'] + eps)
    df_out['autoFE_f_13'] = df_out['synergy_food_and_drink_1000m'] / (df_out['dist_nearest_park'] + eps)
    df_out['autoFE_f_14'] = df_out['synergy_lifestyle_services_1000m'] / (df_out['synergy_lodging_1000m'] + eps)

    # # 4. Complex GroupBy Logic
    # # GroupByThenRank: Groups by government synergy, then ranks the food synergy within that specific group
    # df_out['autoFE_f_5'] = df_out.groupby('synergy_community_and_government_1000m')['synergy_food_and_drink_1000m'].rank(method='average')
    
    # # GroupByThenMax
    # df_out['autoFE_f_11'] = df_out.groupby('synergy_sports_and_recreation_1000m')['dist_nearest_water'].transform('max')

    return df_out

# Usage Example:
# X_train = generate_top_15_features(X_train)
df=generate_top_15_features(df)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from category_encoders import CatBoostEncoder
import joblib

# 1. CONVERT LOW-CARDINALITY FEATURES TO 'CATEGORY' (Do this before splitting)
df['nearest_road_class'] = df['nearest_road_class'].astype('category')

# If you still have nearest_road_surface, uncomment the line below:
# df['nearest_road_surface'] = df['nearest_road_surface'].astype('category')

# 2. SEPARATE FEATURES AND TARGET
X = df.drop(columns=['viability_score_0_1'])
y = df['viability_score_0_1']

# 3. SPLIT THE DATA (70% Train, 15% Val, 15% Test)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# 4. TARGET ENCODE THE HIGH-CARDINALITY FEATURE (Do this after splitting)
# Initialize the encoder specifically for the target_category column
encoder = CatBoostEncoder(cols=['target_category'])

# FIT AND TRANSFORM ON TRAIN: The encoder learns the averages from the training data
X_train = encoder.fit_transform(X_train, y_train)

# TRANSFORM ON VAL & TEST: The encoder applies the learned averages to the unseen data
X_val = encoder.transform(X_val)
X_test = encoder.transform(X_test)

# 2. Save the CatBoost Encoder (Pickle/Joblib is standard for encoders)
joblib.dump(encoder, 'viability_encoder.pkl')

## XGBoost Hyperparameter training

In [ ]:
# !pip install optuna

In [ ]:
# import optuna
# import xgboost as xgb
# from sklearn.metrics import mean_squared_error
# import numpy as np
# import warnings

# # Suppress XGBoost warnings to keep logs clean
# warnings.filterwarnings("ignore")

# # 1. Define the Objective Function for Optuna
# def objective(trial):
#     # THE MAGIC TRICK: Alternate GPUs based on the trial number
#     # Trial 0 goes to GPU 0, Trial 1 goes to GPU 1, Trial 2 -> GPU 0, etc.
#     gpu_id = trial.number % 2
#     device = f'cuda:{gpu_id}'
    
#     # 2. Suggest hyperparameters to test
#     param = {
#         'enable_categorical': True,
#         'early_stopping_rounds' : 50,
#         'tree_method': 'hist',
#         'device': device,
#         'n_estimators': 3000,          # High limit, let early stopping catch it
#         'max_depth': trial.suggest_int('max_depth', 6, 12),
#         'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
#         'subsample': trial.suggest_float('subsample', 0.6, 1.0),
#         'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
#         'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
#         'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True) # Great for preventing overfitting
#     }

#     # 3. Initialize the model with the suggested params
#     model = xgb.XGBRegressor(**param)
    
#     # 4. Train the model (Turn off verbose so it doesn't spam your notebook)
#     model.fit(
#         X_train, y_train,
#         eval_set=[(X_val, y_val)],
#         verbose=False
#     )
    
#     # 5. Make predictions on the validation set
#     preds = model.predict(X_val)
    
#     # 6. Calculate and return the RMSE
#     rmse = np.sqrt(mean_squared_error(y_val, preds))
    
#     return rmse

# print("🚀 Starting Dual-GPU Optuna Tuning...")

# # 7. Create the Optuna study (We want to minimize RMSE)
# study = optuna.create_study(direction='minimize', study_name="Viability_XGB_Optimization")

# # 8. RUN THE OPTIMIZATION!
# # n_jobs=2 fires up both T4 GPUs at the exact same time!
# # n_trials=30 means it will test 30 different combinations (15 per GPU)
# study.optimize(objective, n_trials=30, n_jobs=2)

# print("\n✅ Tuning Complete!")

In [ ]:
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score

print("1. Downcasting to 32-bit for maximum T4 GPU speed...")
# Float downcast
float_cols = X_train.select_dtypes(include=['float64']).columns
X_train[float_cols] = X_train[float_cols].astype(np.float32)
X_val[float_cols] = X_val[float_cols].astype(np.float32)
X_test[float_cols] = X_test[float_cols].astype(np.float32)

# Int downcast
int_cols = X_train.select_dtypes(include=['int64']).columns
X_train[int_cols] = X_train[int_cols].astype(np.int32)
X_val[int_cols] = X_val[int_cols].astype(np.int32)
X_test[int_cols] = X_test[int_cols].astype(np.int32)

print("2. Pre-compiling data directly into GPU memory...")
dtrain = xgb.QuantileDMatrix(X_train, label=y_train, enable_categorical=True)
dval = xgb.QuantileDMatrix(X_val, label=y_val, ref=dtrain, enable_categorical=True)
dtest = xgb.QuantileDMatrix(X_test, label=y_test, ref=dtrain, enable_categorical=True)

# 3. Your Optuna Winning Parameters
final_params = {
    'max_depth': 12,
    'learning_rate': 0.036936,
    'subsample': 0.870587,
    'colsample_bytree': 0.604681,
    'min_child_weight': 8,
    'gamma': 6.1950719e-08,
    'tree_method': 'hist',
    'device': 'cuda:0',
    'objective': 'reg:squarederror'
}

print("🚀 Launching native Accelerated XGBoost training...")
final_model = xgb.train(
    final_params,
    dtrain,
    num_boost_round=10000,
    evals=[(dval, "Validation")],
    early_stopping_rounds=100,
    verbose_eval=200
)

# 4. Final Evaluation
print("\n📊 Calculating Final Official Metrics on X_test...")
y_pred = final_model.predict(dtest)

final_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
final_r2 = r2_score(y_test, y_pred)

print(f"🔥 FINAL TEST RMSE: {final_rmse:.5f}")
print(f"🔥 FINAL TEST R-SQUARED: {final_r2:.5f}")

## XGBOOST METRIC TESTING & SAVING

In [ ]:
# 1. Save the XGBoost model (JSON is the safest format for XGBoost)
final_model.save_model('xgb_model.json')


import numpy as np
import pandas as pd
from sklearn.metrics import (
    mean_squared_error, 
    mean_absolute_error, 
    r2_score, 
    median_absolute_error,
    max_error,
    explained_variance_score,
    mean_absolute_percentage_error
)

print("📊 Calculating Comprehensive Suite of Regression Metrics...")

# 1. Ensure you have the latest predictions
# y_pred = final_model.predict(dtest)

# 2. Calculate every major metric
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
medae = median_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
max_err = max_error(y_test, y_pred)
evs = explained_variance_score(y_test, y_pred)

# Handle MAPE carefully in case of zero-values in y_test
# Adding a tiny epsilon prevents divide-by-zero errors
epsilon = 1e-10
mape = mean_absolute_percentage_error(y_test + epsilon, y_pred)

# 3. Calculate Adjusted R-Squared
# Penalizes the score if you have too many useless features
n = len(y_test) # Number of rows
p = X_test.shape[1] # Number of features
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

# 4. Compile into a Dictionary
metrics_dict = {
    "R-Squared": round(r2, 5),
    "Adjusted R-Squared": round(adj_r2, 5),
    "RMSE": round(rmse, 5),
    "MSE": round(mse, 5),
    "MAE (Mean Absolute Error)": round(mae, 5),
    "Median Absolute Error": round(medae, 5),
    "MAPE (Percentage Error)": round(mape, 5),
    "Explained Variance": round(evs, 5),
    "Max Error (Worst Prediction)": round(max_err, 5)
}

# 5. Convert to a Pandas DataFrame for easy viewing and saving
metrics_df = pd.DataFrame(list(metrics_dict.items()), columns=['Metric', 'Score'])

# 6. Display to the screen
display(metrics_df)

# 7. Export to Disk for Later Use
metrics_df.to_csv('xgboost_metrics.csv', index=False)

print("✅ All metrics saved to 'xgboost_metrics.csv'")

## CATBOOST TRAINING

In [ ]:
# 1. Automatically find all categorical/object columns (like 'nearest_road_class')
cat_cols = list(X_train.select_dtypes(include=['category', 'object']).columns)
print(f"Categorical features detected: {cat_cols}")

# 2. Re-initialize the model (same as before)
from catboost import CatBoostRegressor

cb_model = CatBoostRegressor(
    iterations=10000,
    learning_rate=0.04,
    depth=8,
    task_type='GPU',
    loss_function='RMSE',
    eval_metric='RMSE',
    random_seed=42
)

print("🚀 Launching CatBoost with explicit categorical handling...")

# 3. Pass the cat_cols into the .fit() method
cb_model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    cat_features=cat_cols,       # <--- The crucial fix
    early_stopping_rounds=100,
    verbose=200
)

## CATBOOST METRIC TESTING & SAVING

In [ ]:
print("1. Saving CatBoost Model...")
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    median_absolute_error, max_error, explained_variance_score,
    mean_absolute_percentage_error
)

def save_and_evaluate(y_true, y_pred, n_features, model_name):
    print(f"\n📊 Calculating Comprehensive Metrics for {model_name}...")
    
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    medae = median_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    max_err = max_error(y_true, y_pred)
    evs = explained_variance_score(y_true, y_pred)
    
    # Handle MAPE carefully
    mape = mean_absolute_percentage_error(y_true + 1e-10, y_pred)
    
    # Adjusted R-Squared
    n = len(y_true)
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - n_features - 1)
    
    metrics_dict = {
        "R-Squared": round(r2, 5),
        "Adjusted R-Squared": round(adj_r2, 5),
        "RMSE": round(rmse, 5),
        "MSE": round(mse, 5),
        "MAE (Mean Absolute Error)": round(mae, 5),
        "Median Absolute Error": round(medae, 5),
        "MAPE (Percentage Error)": round(mape, 5),
        "Explained Variance": round(evs, 5),
        "Max Error (Worst Prediction)": round(max_err, 5)
    }
    
    metrics_df = pd.DataFrame(list(metrics_dict.items()), columns=['Metric', 'Score'])
    display(metrics_df)
    
    # Export to CSV
    file_name = f"{model_name.lower().replace(' ', '_')}_metrics.csv"
    metrics_df.to_csv(file_name, index=False)
    print(f"✅ {model_name} metrics saved to '{file_name}'")

# Number of features for Adjusted R2
n_features = X_test.shape[1]

# ---------------------------------------------------------
# 1. CATBOOST
# ---------------------------------------------------------
# Save the model
cb_model.save_model('catboost_model.cbm')

# GENERATE CATBOOST PREDICTIONS FIRST
print("Generating CatBoost predictions on test set...")
y_pred_cat = cb_model.predict(X_test)

# Evaluate using the new CatBoost predictions
save_and_evaluate(y_test, y_pred_cat, n_features, "CatBoost")

## LGBM TRAINING

In [ ]:
import lightgbm as lgb

lgb_model = lgb.LGBMRegressor(
    n_estimators=10000,
    learning_rate=0.03,
    num_leaves=63,
    subsample=0.85,
    colsample_bytree=0.6,
    device='gpu',
    random_state=42
)
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)]
)

## LGBM METRIC TESTING & SAVING

In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    median_absolute_error, max_error, explained_variance_score,
    mean_absolute_percentage_error
)

def save_and_evaluate(y_true, y_pred, n_features, model_name):
    print(f"\n📊 Calculating Comprehensive Metrics for {model_name}...")
    
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    medae = median_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    max_err = max_error(y_true, y_pred)
    evs = explained_variance_score(y_true, y_pred)
    
    # Handle MAPE carefully
    mape = mean_absolute_percentage_error(y_true + 1e-10, y_pred)
    
    # Adjusted R-Squared
    n = len(y_true)
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - n_features - 1)
    
    metrics_dict = {
        "R-Squared": round(r2, 5),
        "Adjusted R-Squared": round(adj_r2, 5),
        "RMSE": round(rmse, 5),
        "MSE": round(mse, 5),
        "MAE (Mean Absolute Error)": round(mae, 5),
        "Median Absolute Error": round(medae, 5),
        "MAPE (Percentage Error)": round(mape, 5),
        "Explained Variance": round(evs, 5),
        "Max Error (Worst Prediction)": round(max_err, 5)
    }
    
    metrics_df = pd.DataFrame(list(metrics_dict.items()), columns=['Metric', 'Score'])
    display(metrics_df)
    
    # Export to CSV
    file_name = f"{model_name.lower().replace(' ', '_')}_metrics.csv"
    metrics_df.to_csv(file_name, index=False)
    print(f"✅ {model_name} metrics saved to '{file_name}'")

# Number of features for Adjusted R2
n_features = X_test.shape[1]

# ---------------------------------------------------------
# 2. LIGHTGBM
# ---------------------------------------------------------
# Using .booster_ because we trained using the Scikit-Learn API wrapper
lgb_model.booster_.save_model('lightgbm_model.txt')

# GENERATE LIGHTGBM PREDICTIONS FIRST
print("Generating LightGBM predictions on test set...")
y_pred_lgb = lgb_model.predict(X_test)

# Evaluate using the new LightGBM predictions
save_and_evaluate(y_test, y_pred_lgb, n_features, "LightGBM")

## RANDOMFOREST TRAINING

In [ ]:
!pip install cuml-cu12 --extra-index-url=https://pypi.nvidia.com

In [ ]:
import numpy as np
import time

print("1. Encoding Categoricals & Casting to float32...")
X_train_rf = X_train.copy()
X_val_rf = X_val.copy()
X_test_rf = X_test.copy()

# Find the categorical columns
cat_cols = list(X_train.select_dtypes(include=['category', 'object']).columns)

for col in cat_cols:
    # 1. Create the category mapping strictly from the TRAINING data
    train_categories = X_train_rf[col].astype('category').cat.categories
    
    # 2. Apply that exact mapping to all three datasets
    X_train_rf[col] = pd.Categorical(X_train_rf[col], categories=train_categories).codes
    X_val_rf[col] = pd.Categorical(X_val_rf[col], categories=train_categories).codes
    X_test_rf[col] = pd.Categorical(X_test_rf[col], categories=train_categories).codes

# cuML demands float32 for GPU memory optimization
X_train_rf = X_train_rf.astype(np.float32)
y_train_rf = y_train.astype(np.float32)
X_test_rf = X_test_rf.astype(np.float32)
y_test_rf = y_test.astype(np.float32)

print("2. Initializing cuML GPU Random Forest...")
# Import cuML inside the block so it runs after the pip install
from cuml.ensemble import RandomForestRegressor as cuRF
from sklearn.metrics import mean_squared_error, r2_score

# Sub-bagging parameters for high variance reduction
rf_model = cuRF(
    n_estimators=150,
    max_depth=16,
    max_features='sqrt',
    n_bins=256,         # The GPU equivalent of histogram bins
    random_state=42
)

print("3. Training on T4 GPU...")
start_time = time.time()
rf_model.fit(X_train_rf, y_train_rf)
print(f"✅ Training completed in {time.time() - start_time:.2f} seconds!")

print("\n4. Evaluating Random Forest...")
y_pred_rf = rf_model.predict(X_test_rf)

rf_rmse = np.sqrt(mean_squared_error(y_test_rf, y_pred_rf))
rf_r2 = r2_score(y_test_rf, y_pred_rf)

print(f"🔥 RANDOM FOREST FINAL TEST RMSE: {rf_rmse:.5f}")
print(f"🔥 RANDOM FOREST FINAL TEST R-SQUARED: {rf_r2:.5f}")

## RF METRIC TESTING & SAVING

In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    median_absolute_error, max_error, explained_variance_score,
    mean_absolute_percentage_error
)

def save_and_evaluate(y_true, y_pred, n_features, model_name):
    print(f"\n📊 Calculating Comprehensive Metrics for {model_name}...")
    
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    medae = median_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    max_err = max_error(y_true, y_pred)
    evs = explained_variance_score(y_true, y_pred)
    
    # Handle MAPE carefully
    mape = mean_absolute_percentage_error(y_true + 1e-10, y_pred)
    
    # Adjusted R-Squared
    n = len(y_true)
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - n_features - 1)
    
    metrics_dict = {
        "R-Squared": round(r2, 5),
        "Adjusted R-Squared": round(adj_r2, 5),
        "RMSE": round(rmse, 5),
        "MSE": round(mse, 5),
        "MAE (Mean Absolute Error)": round(mae, 5),
        "Median Absolute Error": round(medae, 5),
        "MAPE (Percentage Error)": round(mape, 5),
        "Explained Variance": round(evs, 5),
        "Max Error (Worst Prediction)": round(max_err, 5)
    }
    
    metrics_df = pd.DataFrame(list(metrics_dict.items()), columns=['Metric', 'Score'])
    display(metrics_df)
    
    # Export to CSV
    file_name = f"{model_name.lower().replace(' ', '_')}_metrics.csv"
    metrics_df.to_csv(file_name, index=False)
    print(f"✅ {model_name} metrics saved to '{file_name}'")

# Number of features for Adjusted R2
n_features = X_test.shape[1]

# ---------------------------------------------------------
# 3. RANDOM FOREST
# ---------------------------------------------------------
# RF must be saved via Joblib as it lacks a native C++ format
joblib.dump(rf_model, 'rf_model.pkl')
save_and_evaluate(y_test, y_pred_rf, n_features, "Random Forest")

## COMBINATION TRAINING

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from catboost import CatBoostRegressor
import lightgbm as lgb
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
import joblib

# ---------------------------------------------------------
# 1. LOAD MODELS FROM DISK
# ---------------------------------------------------------
print("1. Loading saved models from disk...")
# XGBoost
xgb_model = xgb.Booster()
xgb_model.load_model('xgb_model.json')

# CatBoost
cb_model = CatBoostRegressor()
cb_model.load_model('catboost_model.cbm')

# LightGBM
lgb_model = lgb.Booster(model_file='lightgbm_model.txt')

# Random Forest
rf_model = joblib.load('rf_model.pkl')

# ---------------------------------------------------------
# 2. REGENERATE PREDICTIONS
# ---------------------------------------------------------
print("2. Generating predictions for Validation and Test sets...")

# Add enable_categorical=True to allow XGBoost to process the 'nearest_road_class' column
dval = xgb.DMatrix(X_val, enable_categorical=True)
dtest = xgb.DMatrix(X_test, enable_categorical=True)

# Validation Predictions (Used to train the Ridge weights)
y_val_xgb = xgb_model.predict(dval)
y_val_cat = cb_model.predict(X_val)
y_val_lgb = lgb_model.predict(X_val)
# IMPORTANT: cuML needs the float32 specific arrays you made earlier
y_val_rf = rf_model.predict(X_val_rf) 

# Test Predictions (Used for the final evaluation)
y_pred_xgb = xgb_model.predict(dtest)
y_pred_cat = cb_model.predict(X_test)
y_pred_lgb = lgb_model.predict(X_test)
y_pred_rf = rf_model.predict(X_test_rf)

# ---------------------------------------------------------
# 3. BUILD THE RIDGE META-MODEL
# ---------------------------------------------------------
print("3. Training Ridge Regressor Meta-Model (XGBoost + CatBoost + LightGBM + Random Forest)...")

# Stack all 4 predictions into a new dataset
meta_X_val = np.column_stack((y_val_xgb, y_val_cat, y_val_lgb, y_val_rf))
meta_X_test = np.column_stack((y_pred_xgb, y_pred_cat, y_pred_lgb, y_pred_rf))

# Train the Meta-Model
meta_model = Ridge(alpha=1.0)
meta_model.fit(meta_X_val, y_val)

# ---------------------------------------------------------
# 4. FINAL EVALUATION
# ---------------------------------------------------------
final_ensemble_preds = meta_model.predict(meta_X_test)
ensemble_rmse = np.sqrt(mean_squared_error(y_test, final_ensemble_preds))
ensemble_r2 = r2_score(y_test, final_ensemble_preds)

print(f"\n🔥 FINAL SUPER-ENSEMBLE TEST RMSE: {ensemble_rmse:.5f}")
print(f"🔥 FINAL SUPER-ENSEMBLE TEST R-SQUARED: {ensemble_r2:.5f}")

print("\n📊 Model Blending Weights (Who the Meta-Model trusts most):")
# Added Random Forest to the label loop
for name, weight in zip(['XGBoost', 'CatBoost', 'LightGBM', 'Random Forest'], meta_model.coef_):
    print(f" -> {name}: {weight:.4f}")

# ---------------------------------------------------------
# 5. SAVE THE META-MODEL FOR THE API
# ---------------------------------------------------------
joblib.dump(meta_model, 'ensemble_ridge_meta_model.pkl')
print("\n✅ Meta-model saved as 'ensemble_ridge_meta_model.pkl' for Modal deployment!")

## COMBINATION METRIC TESTING

In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    median_absolute_error, max_error, explained_variance_score,
    mean_absolute_percentage_error
)

def save_and_evaluate(y_true, y_pred, n_features, model_name):
    print(f"\n📊 Calculating Comprehensive Metrics for {model_name}...")
    
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    medae = median_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    max_err = max_error(y_true, y_pred)
    evs = explained_variance_score(y_true, y_pred)
    
    # Handle MAPE carefully
    mape = mean_absolute_percentage_error(y_true + 1e-10, y_pred)
    
    # Adjusted R-Squared
    n = len(y_true)
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - n_features - 1)
    
    metrics_dict = {
        "R-Squared": round(r2, 5),
        "Adjusted R-Squared": round(adj_r2, 5),
        "RMSE": round(rmse, 5),
        "MSE": round(mse, 5),
        "MAE (Mean Absolute Error)": round(mae, 5),
        "Median Absolute Error": round(medae, 5),
        "MAPE (Percentage Error)": round(mape, 5),
        "Explained Variance": round(evs, 5),
        "Max Error (Worst Prediction)": round(max_err, 5)
    }
    
    metrics_df = pd.DataFrame(list(metrics_dict.items()), columns=['Metric', 'Score'])
    display(metrics_df)
    
    # Export to CSV
    file_name = f"{model_name.lower().replace(' ', '_')}_metrics.csv"
    metrics_df.to_csv(file_name, index=False)
    print(f"✅ {model_name} metrics saved to '{file_name}'")

# Number of features for Adjusted R2
n_features = X_test.shape[1]

# ---------------------------------------------------------
# 4. ENSEMBLE RIDGE META-MODEL EVALUATION
# ---------------------------------------------------------
# Note: The meta_model was already saved as 'ensemble_ridge_meta_model.pkl' in the previous cell.
# We pass 'final_ensemble_preds' which holds our stacked predictions.

print("Generating final comprehensive report for the Super-Ensemble...")
save_and_evaluate(y_test, final_ensemble_preds, n_features, "Super Ensemble")

## SAVING AS A DATASET

In [ ]:
import os

# Hardcode your Kaggle API credentials
os.environ['KAGGLE_USERNAME'] = "sciencekonstant"
os.environ['KAGGLE_KEY'] = "c9f7de65751f1916943353d941cf6687"

print("✅ Kaggle API credentials loaded into environment!")

In [ ]:
import os
import json

# 1. Extract your username to format the dataset ID properly
username = os.environ.get('KAGGLE_USERNAME')
if not username:
    raise ValueError("KAGGLE_USERNAME environment variable is not set!")

dataset_slug = "spatial-viability-models-backup"
dataset_title = "Spatial Viability Optimized Models Backup"

print("1. Generating Kaggle dataset metadata...")
# Initialize the dataset-metadata.json file in the current directory (.)
!kaggle datasets init -p .

# 2. Modify the placeholder metadata with your specific details
with open('dataset-metadata.json', 'r') as f:
    meta = json.load(f)

# The ID must be in the format "username/dataset-slug"
meta['id'] = f"{username}/{dataset_slug}"
meta['title'] = dataset_title

# Write the updated JSON back to disk
with open('dataset-metadata.json', 'w') as f:
    json.dump(meta, f, indent=4)

print(f"2. Metadata configured for dataset: {meta['id']}")
print("3. Uploading working directory to Kaggle Datasets...")

# 3. Create the dataset (this automatically zips the directory and uploads it securely)
# The -u flag ensures it is created as Private so only you can see it
!kaggle datasets create -p . -u -r zip

print("✅ Backup complete! You can view your files in your Kaggle Datasets tab.")